# HOG size and sex bias

Load packages

In [47]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

Add function for Wilson confidence interval, instead of traditional CI, as the proportions for some sizes are 0 or 1. 

In [48]:
def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
    return center - margin, center + margin

#z = z-score corresponding here to 1.96 for 95% interval.  
#p = p(hat) = proportion of success. 
#k = sucesses 
#n = number of trials 

Load the annotated result dataset from Salmon map

In [49]:
salmon_map_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_map_dominance_DE_sex_results_new_filtering_genotype_controlled_age_rank_May27.csv", float_precision='legacy')

# Replace the zeros in padj with 1e-308 avoid log10 issues
salmon_map_full_annot["padj_safe"] = salmon_map_full_annot["padj"].replace(0, 1e-308).fillna(1)

# Add the negative log 10 padj for plotting
salmon_map_full_annot["neglog10_padj"] = -np.log10(salmon_map_full_annot["padj_safe"])

# Add significance to differentially expressed genes 
salmon_map_full_annot["significant"] = (
    (salmon_map_full_annot["padj"] < 0.05) &
    (salmon_map_full_annot["log2FoldChange"].abs() > 1)
)

# Add a label to the significant genes
salmon_map_full_annot["label"] = salmon_map_full_annot["gene_id"].where(salmon_map_full_annot["significant"], "")

# load the full annotation and the lowly expressed datasets for later defining hog size columns 
full_annotation = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/C_mac_full_annotation_with_age_May27.csv")
prefilter_df    = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/prefilter_transcripts_annotated_May27.csv")

salmon_map_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,duplication_node,gene_tree_node,duplication_support,duplication_type,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,220384.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n3,1.0,Terminal,0.7587,0.061748,1.316673e-08,7.880522,False,
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,227675.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n9,1.0,Terminal,0.7587,0.061748,9.353175e-10,9.029041,False,
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,245866.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n13,1.0,Terminal,0.7587,0.061748,2.387530e-17,16.622051,False,
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n8,1.0,Terminal,0.7587,0.061748,2.671748e-04,3.573204,True,g5
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,263441.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.659726e-79,78.436551,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17569,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,NaN,NaN,NaN,4.238130e-02,1.372826,False,
17570,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n9,1.0,Terminal,0.7587,0.061748,6.956768e-04,3.157592,True,g34884
17571,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n43,1.0,Terminal,0.7587,0.061748,4.718367e-08,7.326208,True,g34922
17572,19.121063,2.284212,0.326813,6.989347,2.761695e-12,6.049015e-12,g35167.t1,g35167,utg003885l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.049015e-12,11.218315,True,g35167


Remove entries that does not have a HOG

In [50]:
salmon_map_results_HOG = salmon_map_full_annot.dropna(subset=["HOG"])
salmon_map_results_HOG
# Down from 17.574 to 15.648 transcripts


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,duplication_node,gene_tree_node,duplication_support,duplication_type,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,220384.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n3,1.0,Terminal,0.7587,0.061748,1.316673e-08,7.880522,False,
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,227675.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n9,1.0,Terminal,0.7587,0.061748,9.353175e-10,9.029041,False,
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,245866.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n13,1.0,Terminal,0.7587,0.061748,2.387530e-17,16.622051,False,
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n8,1.0,Terminal,0.7587,0.061748,2.671748e-04,3.573204,True,g5
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,263441.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.659726e-79,78.436551,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17567,10.439363,0.527491,0.759882,0.694175,4.875723e-01,5.260053e-01,g34647.t1,g34647,utg003542l,22333.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n2,1.0,Terminal,0.7587,0.061748,5.260053e-01,0.279010,False,
17569,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,NaN,NaN,NaN,4.238130e-02,1.372826,False,
17570,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n9,1.0,Terminal,0.7587,0.061748,6.956768e-04,3.157592,True,g34884
17571,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,C_maculatus_filtered_proteinfasta_TE_filtered,n43,1.0,Terminal,0.7587,0.061748,4.718367e-08,7.326208,True,g34922


Compute number of paralogs in each HOG.  
Add the hog_size columns to the results table by merging on HOG name from the two other datasets. 

In [51]:
# HOG size defined at three levels
hog_size_genome  = full_annotation.dropna(subset=["HOG"]).groupby("HOG").size().reset_index(name="hog_size_genome")
hog_size_mapped  = prefilter_df.dropna(subset=["HOG"]).groupby("HOG").size().reset_index(name="hog_size_mapped")
hog_size_expressed = salmon_map_results_HOG.groupby("HOG").size().reset_index(name="hog_size_expressed")

# Merge all three onto salmon_map_results_HOG
salmon_map_results_HOG = salmon_map_results_HOG.merge(hog_size_genome,   on="HOG", how="left")
salmon_map_results_HOG = salmon_map_results_HOG.merge(hog_size_mapped,   on="HOG", how="left")
salmon_map_results_HOG = salmon_map_results_HOG.merge(hog_size_expressed, on="HOG", how="left")

summary = pd.DataFrame({
    "level":        ["genome", "mapped", "expressed"],
    "n_HOGs":       [hog_size_genome.shape[0], hog_size_mapped.shape[0], hog_size_expressed.shape[0]],
    "median_size":  [hog_size_genome["hog_size_genome"].median(),
                     hog_size_mapped["hog_size_mapped"].median(),
                     hog_size_expressed["hog_size_expressed"].median()],
    "max_size":     [hog_size_genome["hog_size_genome"].max(),
                     hog_size_mapped["hog_size_mapped"].max(),
                     hog_size_expressed["hog_size_expressed"].max()],
})
print(summary)

print("\n sanity check: ")

for name, df in [("full_annotation", full_annotation),
                 ("prefilter_df",    prefilter_df),
                 ("de_results",      salmon_map_full_annot)]:
    print(f"{name}: {df['HOG'].notna().sum():,} with HOG / {len(df):,} total")

salmon_map_results_HOG


       level  n_HOGs  median_size  max_size
0     genome   14402          1.0        92
1     mapped   14402          1.0        78
2  expressed   10807          1.0        28

 sanity check: 
full_annotation: 31,408 with HOG / 37,988 total
prefilter_df: 29,943 with HOG / 36,382 total
de_results: 15,594 with HOG / 17,574 total


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,duplication_type,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label,hog_size_genome,hog_size_mapped,hog_size_expressed
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,220384.0,...,Terminal,0.7587,0.061748,1.316673e-08,7.880522,False,,2,2,2
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,227675.0,...,Terminal,0.7587,0.061748,9.353175e-10,9.029041,False,,2,2,2
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,245866.0,...,Terminal,0.7587,0.061748,2.387530e-17,16.622051,False,,2,2,2
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,Terminal,0.7587,0.061748,2.671748e-04,3.573204,True,g5,2,2,2
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,263441.0,...,NaN,NaN,NaN,3.659726e-79,78.436551,False,,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15589,10.439363,0.527491,0.759882,0.694175,4.875723e-01,5.260053e-01,g34647.t1,g34647,utg003542l,22333.0,...,Terminal,0.7587,0.061748,5.260053e-01,0.279010,False,,23,21,8
15590,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,4.238130e-02,1.372826,False,,2,2,2
15591,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,Terminal,0.7587,0.061748,6.956768e-04,3.157592,True,g34884,23,23,9
15592,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,Terminal,0.7587,0.061748,4.718367e-08,7.326208,True,g34922,22,21,8


In the genome (and mapped) there are 14 563 gene families. The represented in the expressed dataset is 10 850. The remaining 3 713 are not visible as they are not expressed

Add sex bias column

In [52]:
salmon_map_results_HOG["sex_bias"] = "not_significant"

salmon_map_results_HOG.loc[
    salmon_map_results_HOG["significant"] &
    (salmon_map_results_HOG["log2FoldChange"] > 0),
    "sex_bias"
] = "male"

salmon_map_results_HOG.loc[
    salmon_map_results_HOG["significant"] &
    (salmon_map_results_HOG["log2FoldChange"] < 0),
    "sex_bias"
] = "female"
salmon_map_results_HOG

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label,hog_size_genome,hog_size_mapped,hog_size_expressed,sex_bias
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,220384.0,...,0.7587,0.061748,1.316673e-08,7.880522,False,,2,2,2,not_significant
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,227675.0,...,0.7587,0.061748,9.353175e-10,9.029041,False,,2,2,2,not_significant
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,245866.0,...,0.7587,0.061748,2.387530e-17,16.622051,False,,2,2,2,not_significant
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,0.7587,0.061748,2.671748e-04,3.573204,True,g5,2,2,2,male
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,263441.0,...,NaN,NaN,3.659726e-79,78.436551,False,,1,1,1,not_significant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15589,10.439363,0.527491,0.759882,0.694175,4.875723e-01,5.260053e-01,g34647.t1,g34647,utg003542l,22333.0,...,0.7587,0.061748,5.260053e-01,0.279010,False,,23,21,8,not_significant
15590,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,4.238130e-02,1.372826,False,,2,2,2,not_significant
15591,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,0.7587,0.061748,6.956768e-04,3.157592,True,g34884,23,23,9,male
15592,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,0.7587,0.061748,4.718367e-08,7.326208,True,g34922,22,21,8,male


In [53]:
def hog_size_table(df, size_col):
    unique_hogs = df.dropna(subset=[size_col]).drop_duplicates(subset="HOG")
    t = (
        unique_hogs.groupby(size_col)
        .size()
        .reset_index(name="n_HOGs")
        .rename(columns={size_col: "HOG_size"})
    )
    t["n_transcripts"] = t["HOG_size"] * t["n_HOGs"]
    return t.set_index("HOG_size")

genome   = hog_size_table(salmon_map_results_HOG, "hog_size_genome")
mapped   = hog_size_table(salmon_map_results_HOG, "hog_size_mapped")
expressed = hog_size_table(salmon_map_results_HOG, "hog_size_expressed")

hog_size_comparison = pd.concat(
    [genome, mapped, expressed],
    axis=1,
    keys=["genome", "mapped", "expressed"]
).fillna(0).astype(int)

hog_size_comparison.columns = [
    "Genome gene families",   "Genome transcripts",
    "Mapped gene families",        "Mapped transcripts",
    "Expressed gene families",     "Expressed transcripts",
]

print(sum(hog_size_comparison["Genome gene families"]))
print(sum(hog_size_comparison["Genome transcripts"]))
print(sum(hog_size_comparison["Mapped gene families"]))
print(sum(hog_size_comparison["Mapped transcripts"]))
print(sum(hog_size_comparison["Expressed gene families"]))
print(sum(hog_size_comparison["Expressed transcripts"]))

hog_size_comparison


10807
21508
10807
21043
10807
15594


,Genome gene families,Genome transcripts,Mapped gene families,Mapped transcripts,Expressed gene families,Expressed transcripts
HOG_size,,,,,,
1,7545,7545,7606,7606,8157,8157
2,1914,3828,1884,3768,1839,3678
3,463,1389,452,1356,386,1158
4,224,896,222,888,175,700
5,123,615,124,620,80,400
6,92,552,94,564,52,312
7,63,441,61,427,31,217
8,71,568,68,544,27,216
9,36,324,35,315,17,153


Table above shows the expressed transcripts only, we have 10 850 gene families and 15 648 transcripts represented. But in reality there are 21 494 annotated transcripts in the genome, and  21030 in Salmon mapped calculated by HOG_size*n_HOGs 

# Analysis 1: Bias direction among biased transcripts. 
Which fraction is male within each HOG size? 

Which transcripts that belong to a HOG is biased? Remove the unbiased transcripts

In [54]:
biased = salmon_map_results_HOG[
    salmon_map_results_HOG["sex_bias"].isin(["male", "female"])
].copy()

biased
# 6472 belong to a HOG and are significantly sex biased

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,node_depth_from_root,branch_length,padj_safe,neglog10_padj,significant,label,hog_size_genome,hog_size_mapped,hog_size_expressed,sex_bias
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,257697.0,...,0.758700,0.061748,2.671748e-04,3.573204,True,g5,2,2,2,male
5,70.718212,-1.039030,0.089153,-11.654489,2.176813e-31,7.787918e-31,g7.t1,g7,utg000001l,371755.0,...,NaN,NaN,7.787918e-31,30.108579,True,g7,1,1,1,female
6,475.721002,1.120124,0.059428,18.848397,3.028815e-79,2.541218e-78,g8.t1,g8,utg000001l,510966.0,...,NaN,NaN,2.541218e-78,77.594958,True,g8,1,1,1,male
7,97.175715,1.341896,0.124766,10.755269,5.597042e-27,1.811622e-26,g9.t1,g9,utg000001l,531498.0,...,0.696952,0.035914,1.811622e-26,25.741932,True,g9,2,2,2,male
11,237.620281,-4.015786,0.186764,-21.501956,1.492615e-102,1.902623e-101,g13.t1,g13,utg000001l,666271.0,...,0.696952,0.035914,1.902623e-101,100.720647,True,g13,1,1,1,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15585,3.429476,2.064888,0.534763,3.861312,1.127796e-04,1.773450e-04,g34258.t1,g34258,utg003361l,24503.0,...,0.758700,0.061748,1.773450e-04,3.751181,True,g34258,13,13,3,male
15588,3.024122,2.592433,0.660156,3.927001,8.601149e-05,1.363904e-04,g34611.t1,g34611,utg003498l,55900.0,...,NaN,NaN,1.363904e-04,3.865216,True,g34611,2,2,1,male
15591,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,19710.0,...,0.758700,0.061748,6.956768e-04,3.157592,True,g34884,23,23,9,male
15592,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,1.0,...,0.758700,0.061748,4.718367e-08,7.326208,True,g34922,22,21,8,male


Count male and female biased transcripts per HOG size.  
Filter out HOG size = 1 as those have no paralogs

In [55]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

hog_counts = (
    biased
    .groupby(["hog_size_genome", "sex_bias"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

hog_counts

sex_bias,hog_size_genome,female,male
0,1,1099,1658
1,2,486,954
2,3,130,395
3,4,67,212
4,5,35,162
5,6,23,162
6,7,43,81
7,8,27,101
8,9,14,48
9,10,12,72


In [56]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''
hog_counts = hog_counts[hog_counts["hog_size_genome"] >= 2].copy()
hog_counts

sex_bias,hog_size_genome,female,male
1,2,486,954
2,3,130,395
3,4,67,212
4,5,35,162
5,6,23,162
6,7,43,81
7,8,27,101
8,9,14,48
9,10,12,72
10,11,10,74


Im doing this as a binomial proportion of p_male = male/(male+female) as some sizes have 0 female biased transcripts.  
Confidence intervals calulated with Wilson score interval, performs better when proportions are close to 0 or 1. 

In [57]:
hog_counts["total_biased"] = hog_counts["male"] + hog_counts["female"]

hog_counts["p_male"] = (
    hog_counts["male"] /
    hog_counts["total_biased"]
)

ci_bounds = hog_counts.apply(
    lambda row: wilson_ci(row["male"], row["total_biased"]),
    axis=1
)

hog_counts["ci_lower"] = [c[0] for c in ci_bounds]
hog_counts["ci_upper"] = [c[1] for c in ci_bounds]

#clipping to avoid floating point residues 
hog_counts["ci_lower"] = hog_counts["ci_lower"].clip(lower=0)
hog_counts["ci_upper"] = hog_counts["ci_upper"].clip(upper=1)


#old normal confidence interval
#hog_counts["se"] = np.sqrt(
#    hog_counts["p_male"] *
#    (1 - hog_counts["p_male"]) /
#    hog_counts["total_biased"]
#)

#hog_counts["ci_lower"] = hog_counts["p_male"] - 1.96 * hog_counts["se"]
#hog_counts["ci_upper"] = hog_counts["p_male"] + 1.96 * hog_counts["se"]
hog_counts


sex_bias,hog_size_genome,female,male,total_biased,p_male,ci_lower,ci_upper
1,2,486,954,1440,0.662500,0.637673,0.686462
2,3,130,395,525,0.752381,0.713714,0.787381
3,4,67,212,279,0.759857,0.706419,0.806236
4,5,35,162,197,0.822335,0.762948,0.869391
5,6,23,162,185,0.875676,0.820356,0.915710
6,7,43,81,124,0.653226,0.565989,0.731254
7,8,27,101,128,0.789062,0.710492,0.850788
8,9,14,48,62,0.774194,0.655941,0.860449
9,10,12,72,84,0.857143,0.766697,0.916351
10,11,10,74,84,0.880952,0.794549,0.934035


Plot

In [58]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

fig1 = px.line(
    hog_counts,
    x="hog_size_genome",
    y="p_male",
    markers=True,
    hover_data={
        "male": True,
        "female": True,
        "total_biased": True,
        "p_male": ":.3f"
    }
)

fig1.update_traces(
    error_y=dict(
        type="data",
        symmetric=False,
        array=hog_counts["ci_upper"] - hog_counts["p_male"],
        arrayminus=hog_counts["p_male"] - hog_counts["ci_lower"]
    )
)

fig1.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="black"
)

fig1.update_layout(
    title=dict(text="Proportion of male biased transcripts in each (genom-wide) gene family size", x=0.5, xanchor="center"),
    xaxis_title="Gene family size (≥2)",
    yaxis_title="Proportion (p_male = #male/#male+#female)",
)

fig1.show()

# Analysis 2: Male/Female/Unbiased prop within HOG size.  

In [59]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

#number in each category
hog_bias = (
    salmon_map_results_HOG
    .groupby(["hog_size_genome", "sex_bias"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

#add the total
hog_bias["total"] = (
    hog_bias["male"] +
    hog_bias["female"] +
    hog_bias["not_significant"]
)


hog_bias

sex_bias,hog_size_genome,female,male,not_significant,total
0,1,1099,1658,4788,7545
1,2,486,954,2067,3507
2,3,130,395,573,1098
3,4,67,212,345,624
4,5,35,162,170,367
5,6,23,162,138,323
6,7,43,81,82,206
7,8,27,101,113,241
8,9,14,48,63,125
9,10,12,72,77,161


Filter out HOG size 1

In [60]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

hog_bias = hog_bias[hog_bias["hog_size_genome"] >= 2].copy()
hog_bias

sex_bias,hog_size_genome,female,male,not_significant,total
1,2,486,954,2067,3507
2,3,130,395,573,1098
3,4,67,212,345,624
4,5,35,162,170,367
5,6,23,162,138,323
6,7,43,81,82,206
7,8,27,101,113,241
8,9,14,48,63,125
9,10,12,72,77,161
10,11,10,74,47,131


Convert to long format instead of wide format

In [61]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''

hog_bias_long = hog_bias.melt(
    id_vars=["hog_size_genome", "total"],
    value_vars=["male", "female", "not_significant"],
    var_name="sex_bias",
    value_name="transcript_count"
)
hog_bias_long

,hog_size_genome,total,sex_bias,transcript_count
0,2,3507,male,954
1,3,1098,male,395
2,4,624,male,212
3,5,367,male,162
4,6,323,male,162
...,...,...,...,...
154,58,7,not_significant,5
155,60,28,not_significant,16
156,62,23,not_significant,13
157,80,23,not_significant,14


Add the proprtions and confidence intervals.  
Confidence intervals calulated with Wilson score interval, performs better when proportions are close to 0 or 1. 

In [62]:
hog_bias_long["proportion"] = (
    hog_bias_long["transcript_count"] /
    hog_bias_long["total"]
)

ci_bounds = hog_bias_long.apply(
    lambda row: wilson_ci(row["transcript_count"], row["total"]),
    axis=1
)

hog_bias_long["ci_lower"] = [c[0] for c in ci_bounds]
hog_bias_long["ci_upper"] = [c[1] for c in ci_bounds]

#clipping to avoid floating point residues 
hog_bias_long["ci_lower"] = hog_bias_long["ci_lower"].clip(lower=0)
hog_bias_long["ci_upper"] = hog_bias_long["ci_upper"].clip(upper=1)


#old normal confidence interval
#hog_bias_long["se"] = np.sqrt(
#    hog_bias_long["proportion"] *
#    (1 - hog_bias_long["proportion"]) /
#    hog_bias_long["total"]
#)

#hog_bias_long["ci_lower"] = hog_bias_long["proportion"] - 1.96 * hog_bias_long["se"]
#hog_bias_long["ci_upper"] = hog_bias_long["proportion"] + 1.96 * hog_bias_long["se"]
hog_bias_long

,hog_size_genome,total,sex_bias,transcript_count,proportion,ci_lower,ci_upper
0,2,3507,male,954,0.272027,0.257554,0.286999
1,3,1098,male,395,0.359745,0.331892,0.388576
2,4,624,male,212,0.339744,0.303663,0.377785
3,5,367,male,162,0.441417,0.391481,0.492567
4,6,323,male,162,0.501548,0.447323,0.555737
...,...,...,...,...,...,...,...
154,58,7,not_significant,5,0.714286,0.358929,0.917783
155,60,28,not_significant,16,0.571429,0.390705,0.734917
156,62,23,not_significant,13,0.565217,0.368110,0.743656
157,80,23,not_significant,14,0.608696,0.407852,0.778426


Plot with plotly

In [63]:
'''
SWITCH BACK TO hog_size_expressed to get the old plot back!
'''
fig2 = px.line(
    hog_bias_long,
    x="hog_size_genome",
    y="proportion",
    color="sex_bias",
    markers=True,
        hover_data={
        "transcript_count": True,
        "total": True,
        "proportion": ":.3f"
    }
)

for trace in fig2.data:
    bias_type = trace.name
    
    subset = hog_bias_long[hog_bias_long["sex_bias"] == bias_type]
    
    trace.error_y = dict(
        type="data",
        symmetric=False,
        array=subset["ci_upper"].values - subset["proportion"].values,
        arrayminus=subset["proportion"].values - subset["ci_lower"].values
    )

fig2.update_layout(
    title=dict(text="Proportion of transcript bias in each gene family size (genome-wide)", x=0.5, xanchor="center"),
    xaxis_title="Gene family size (≥2)",
    yaxis_title="Proportion of transcripts",
)
fig2.show()

# Analysis 3: Variance vs HOG Size
How variable are the log2FoldChange values inside each HOG size? 

Transcript level log2FC variance per HOG size

In [64]:
#this we still want as expressed
#About the expression patterns 
variance_by_size = (
    salmon_map_results_HOG
    .groupby("hog_size_expressed")["log2FoldChange"]
    .agg(
        variance="var",
        n="count"
    )
    .reset_index()
)

#filter out size 1
variance_by_size = variance_by_size[
    variance_by_size["hog_size_expressed"] >= 2
].copy()

#filter out sizes with 0 transcripts
variance_by_size = variance_by_size[
    variance_by_size["n"] > 0
]


variance_by_size

,hog_size_expressed,variance,n
1,2,7.008420,3678
2,3,8.771289,1158
3,4,8.025901,700
4,5,5.760277,400
5,6,4.429956,312
6,7,6.628334,217
7,8,8.496196,216
8,9,4.088329,153
9,10,8.774700,80
10,11,6.777326,88


Plot

In [65]:
#this we stil lwant as expressed
fig3 = px.line(
    variance_by_size,
    x="hog_size_expressed",
    y="variance",
    markers=True,
    hover_data={
        "n": True,
        "variance": ":.3f"
    }
)

fig3.update_layout(
     title=dict(text="Transcript variance within each (expressed) gene family size", x=0.5, xanchor="center"),
    xaxis_title="(Expressed) Gene family size (≥2)",
    yaxis_title="Variance of log2FoldChange",
)

fig3.show()


Expression differene decreases as the family size increases. But for HOG sizes larger than 15 we have very small sample sizes so variance is noisy and unreliable as small sample makes variance unstable

# Analysis 4: Variance within each HOG

In [66]:
#This we stil lwant as expressed
variance_within_hog = (
    salmon_map_results_HOG
    .groupby(["HOG", "hog_size_genome"])["log2FoldChange"]
    .agg(
        variance="var"
    )
    .reset_index()
)

variance_within_hog = variance_within_hog[
    variance_within_hog["hog_size_genome"] >= 2
]

variance_within_hog

,HOG,hog_size_genome,variance
0,N0.HOG0000009,27,1.635579
1,N0.HOG0000011,33,0.324677
2,N0.HOG0000014,56,1.452074
3,N0.HOG0000015,2,0.003742
5,N0.HOG0000017,3,NaN
...,...,...,...
10798,N0.HOG0023741,2,0.615421
10799,N0.HOG0023742,2,1.677436
10800,N0.HOG0023745,2,0.142166
10801,N0.HOG0023750,2,8.145673


Plot

In [67]:
#separate the outlier 
low = variance_within_hog[variance_within_hog["variance"] <= 120]
high = variance_within_hog[variance_within_hog["variance"] > 120]

#create two subplots 
fig4 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.1, 0.9],  # small top, big bottom
    vertical_spacing=0.05
)

#Add bottom panel (main plot)
fig4.add_trace(
    go.Scatter(
        x=low["hog_size_genome"],
        y=low["variance"],
        mode="markers",
        hovertext=low["HOG"],
        name="Variance"
    ),
    row=2, col=1
)

#Add top panel (outlier)
fig4.add_trace(
    go.Scatter(
        x=high["hog_size_genome"],
        y=high["variance"],
        mode="markers",
        hovertext=high["HOG"],
        name="Outlier"
    ),
    row=1, col=1
)

#set axis rate
fig4.update_yaxes(range=[0,120], row=2, col=1)
fig4.update_yaxes(range=[320,340], 
                  tickmode="array",
                  tickvals=[320,340],
                  row=1, col=1)

fig4.update_layout(
    title=dict(text="Within-family variance across gene family sizes", x=0.5, xanchor="center"),
    height=600,
    width=1200,
    showlegend=False,
)
# Set bottom axis titles
fig4.update_xaxes(range=[0,90], title_text=" Gene family size (genome-wide, size ≥ 2)", row=2, col=1, tickmode="linear", dtick=5)
fig4.update_yaxes(title_text="Variance (log2FoldChange)", row=2, col=1)

#top x axis ticks
fig4.update_xaxes(range=[0,90], row=1, col=1, tickmode="linear", dtick=5)
fig4.write_image("within_family_variance_size.svg")
fig4.show()

In [68]:

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=variance_within_hog["hog_size_genome"],
        y=variance_within_hog["variance"],
        mode="markers",
        hovertext=variance_within_hog["HOG"],
        name="Variance"
    )
)

# Layout
fig.update_layout(
    title=dict(text="Within-family variance across gene family sizes", x=0.5, xanchor="center"),
    height=600,
    width=1200,
    showlegend=False
)

# Axes
fig.update_xaxes(
    title_text="Gene family size (genome-wide, size ≥ 2)",
    range=[0, 120],
    tickmode="linear",
    dtick=5
)

fig.update_yaxes(
    title_text="Variance (log2FoldChange)"
)

# Save with new filename
#fig.write_image("within_family_variance_size_simple.svg")

fig.show()

"estimate expression in each sex"  
pick a transcript sequence and paste in conserved domain search (ncbi) to see if it has a TE sequence

# Analysis 5: Within HOG directional bias 

Put the HOGs into categories (biased_sets)

In [69]:
# this we still want as expressed
hog_direction = (
    salmon_map_results_HOG
    .groupby(["HOG", "hog_size_expressed"])["sex_bias"]
    .apply(lambda x: set(x))
    .reset_index()
)

hog_direction.rename(columns={"sex_bias": "bias_set"}, inplace=True)

#classify the sets
def classify_bias(bias_set):
    
    if bias_set == {"male"}:
        return "All male biased"
    
    elif bias_set == {"female"}:
        return "All female biased"
    
    elif bias_set == {"not_significant"}:
        return "All unbiased"
    
    elif bias_set == {"male", "female"}:
        return "Male + Female"
    
    elif bias_set == {"male", "not_significant"}:
        return "Male + Unbiased"
    
    elif bias_set == {"female", "not_significant"}:
        return "Female + Unbiased"
    
    elif bias_set == {"male", "female", "not_significant"}:
        return "All three"
    
    else:
        return "Other"
    
hog_direction["category"] = hog_direction["bias_set"].apply(classify_bias)
hog_direction = hog_direction[
    hog_direction["hog_size_expressed"] >= 2
]
hog_direction = hog_direction.merge(
    salmon_map_results_HOG[["HOG", "hog_size_genome", "hog_size_mapped"]].drop_duplicates("HOG"),
    on="HOG", how="left"
)

hog_direction

,HOG,hog_size_expressed,bias_set,category,hog_size_genome,hog_size_mapped
0,N0.HOG0000009,9,"{male, not_significant}",Male + Unbiased,27,26
1,N0.HOG0000011,7,"{male, not_significant}",Male + Unbiased,33,33
2,N0.HOG0000014,17,"{male, not_significant}",Male + Unbiased,56,56
3,N0.HOG0000015,2,{female},All female biased,2,2
4,N0.HOG0000020,5,{female},All female biased,5,5
...,...,...,...,...,...,...
2645,N0.HOG0023737,2,"{male, not_significant}",Male + Unbiased,2,2
2646,N0.HOG0023741,2,"{female, not_significant}",Female + Unbiased,2,2
2647,N0.HOG0023742,2,"{female, not_significant}",Female + Unbiased,2,2
2648,N0.HOG0023745,2,{not_significant},All unbiased,2,2


Count categories

In [70]:
category_order = [
    "All three",
    "All male biased",
    "Male + Unbiased",
    "Male + Female",
    "Female + Unbiased",
    "All female biased",
    "All unbiased"
]

category_colors = {
    "All three":        "#B39DDB",
    "All male biased":  "#1F4BFF",
    "Male + Unbiased":  "#4A90E2",
    "Male + Female":    "#C77DFF",   
    "Female + Unbiased":"#F28B82",
    "All female biased":"#D32F2F",
    "All unbiased":     "#66BB6A",
}

hog_direction["category"] = pd.Categorical(
    hog_direction["category"],
    categories=category_order,
    ordered=True
)

hog_category_counts = (
    hog_direction
    .groupby("category", observed=False)
    .size()
    .reset_index(name="count")
)

hog_category_counts

,category,count
0,All three,55
1,All male biased,534
2,Male + Unbiased,540
3,Male + Female,38
4,Female + Unbiased,185
5,All female biased,201
6,All unbiased,1097


Plot

In [71]:
fig5 = px.bar(
    hog_category_counts,
    x="category",
    y="count",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order}
)

fig5.update_layout(
    title=dict(text="Number of Gene families in different transcript bias composition categories", x=0.5, xanchor="center"),
    xaxis_title="Gene family composition category",
    yaxis_title="Number of Gene families",
    xaxis_tickangle=45,
    showlegend=False
)

fig5.show()

Get the proportions of bias in each category bar

In [72]:
# Merge category back onto transcript level
transcript_cats = salmon_map_results_HOG.merge(
    hog_direction[["HOG", "category"]],
    on="HOG",
    how="inner"
)

# Count transcripts per category and sex_bias
transcript_composition = (
    transcript_cats
    .groupby(["category", "sex_bias"], observed=False)
    .size()
    .reset_index(name="count")
)

totals = transcript_composition.groupby("category", observed=False)["count"].sum().reset_index(name="total")
transcript_composition = transcript_composition.merge(totals, on="category")
transcript_composition["pct"] = transcript_composition["count"] / transcript_composition["total"] * 100

BIAS_COLORS_3 = {
    "male":            "#6BAED6",
    "female":          "#E07B8A",
    "not_significant": "#708090",
}

bias_order = ["male", "not_significant", "female"]
bias_labels = {
    "male":            "Male-biased",
    "female":          "Female-biased",
    "not_significant": "Unbiased",
}

# Number of HOGs per category
n_hogs_per_category = (
    hog_direction
    .groupby("category", observed=False)
    .size()
    .reset_index(name="n_hogs")
)

# Use transcript proportions per category to split bars
cat_bias = (
    transcript_cats
    .groupby(["category", "sex_bias"], observed=True)
    .size()
    .reset_index(name="count")
)
cat_totals = cat_bias.groupby("category", observed=True)["count"].sum().reset_index(name="total")
cat_bias = cat_bias.merge(cat_totals, on="category")
cat_bias["pct"] = cat_bias["count"] / cat_bias["total"]

# Multiply proportion by n_hogs so bars reach correct height
cat_bias = cat_bias.merge(n_hogs_per_category, on="category")
cat_bias["hog_count"] = cat_bias["pct"] * cat_bias["n_hogs"]

fig5b = go.Figure()

for bias in bias_order:
    sub = cat_bias[cat_bias["sex_bias"] == bias]

    counts_list, texts, customdata = [], [], []
    for cat in category_order:
        row = sub[sub["category"] == cat]
        if len(row) > 0:
            n = row.iloc[0]["hog_count"]
            total = row.iloc[0]["n_hogs"]
            pct = row.iloc[0]["pct"] * 100
            counts_list.append(n)
            texts.append(f"{pct:.1f}%" if pct >= 5 else "")
            customdata.append([round(n), total, round(pct, 1)])
        else:
            counts_list.append(0)
            texts.append("")
            customdata.append([0, 0, 0])

    fig5b.add_trace(go.Bar(
        x=category_order,
        y=counts_list,
        name=bias_labels[bias],
        marker_color=BIAS_COLORS_3[bias],
        marker_line_color="black",
        marker_line_width=0.8,
        text=texts,
        textposition="inside",
        textfont=dict(size=10, color="white", family="Arial Black"),
        customdata=customdata,
        hovertemplate=(
            f"<b>{bias_labels[bias]}</b><br>"
            "Category: %{x}<br>"
            "Proportion: %{customdata[2]}%<br>"
            "Equivalent gene families: %{customdata[0]}<br>"
            "Total gene families in category: %{customdata[1]}<extra></extra>"
        ),
    ))

# n= HOGs per category — positioned just above each bar
for cat in category_order:
    total = n_hogs_per_category[n_hogs_per_category["category"] == cat]["n_hogs"].iloc[0]
    fig5b.add_annotation(
        x=cat,
        y=total,
        yref="y",
        text=f"(n={total})",
        showarrow=False,
        yshift=8,
        font=dict(size=9, color="black"),
        yanchor="bottom",
    )

fig5b.update_layout(
    plot_bgcolor="white",
    barmode="stack",
    title=dict(
        text=(
            "<b>Gene family bias composition by category</b>"
            "<br><sup>Y-axis = number of gene families · "
            "bars split by transcript bias proportion within each category</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Gene family composition category",
        tickangle=45,
        showgrid=False,
        categoryorder="array",
        categoryarray=category_order,
    ),
    yaxis=dict(
        title="Number of gene families",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=False,
        range=[0, n_hogs_per_category["n_hogs"].max() * 1.12],
    ),
    legend=dict(
        title="Sex bias",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
    margin=dict(b=100, t=100),
    width=1400,
    height=800,
)
fig5b.write_image("gene_family_bias_categories.svg")
fig5b.show()

Export categories to R for subsetted mixed model analysis

In [ ]:
# Export HOG category assignments for R
# Keep HOG, category, and size variables for joining to model_data

export_cols = ["HOG", "category", "hog_size_expressed", 
               "hog_size_genome", "hog_size_mapped"]

hog_direction_export = hog_direction[export_cols].copy()

# Confirm what you are exporting
print("Rows:", len(hog_direction_export))
print("Categories:\n", hog_direction_export["category"].value_counts())

#hog_direction_export.to_csv("hog_direction_categories.csv", index=False)
print("Saved to hog_direction_categories.csv")

hog_direction_export

Rows: 2650
Categories:
 category
All unbiased         1097
Male + Unbiased       540
All male biased       534
All female biased     201
Female + Unbiased     185
All three              55
Male + Female          38
Name: count, dtype: int64
Saved to hog_direction_categories.csv


,HOG,category,hog_size_expressed,hog_size_genome,hog_size_mapped
0,N0.HOG0000009,Male + Unbiased,9,27,26
1,N0.HOG0000011,Male + Unbiased,7,33,33
2,N0.HOG0000014,Male + Unbiased,17,56,56
3,N0.HOG0000015,All female biased,2,2,2
4,N0.HOG0000020,All female biased,5,5,5
...,...,...,...,...,...
2645,N0.HOG0023737,Male + Unbiased,2,2,2
2646,N0.HOG0023741,Female + Unbiased,2,2,2
2647,N0.HOG0023742,Female + Unbiased,2,2,2
2648,N0.HOG0023745,All unbiased,2,2,2


Plotly has an issue with the box plots, so here i use go.Figure() to force plotly  

In [74]:
fig6 = go.Figure()

cat_positions = {cat: i for i, cat in enumerate(category_order)}
np.random.seed(42)

for cat in category_order:
    subset = hog_direction[hog_direction["category"] == cat]
    if subset.empty:
        continue

    color  = category_colors[cat]
    x_pos  = cat_positions[cat]

    # Box (no built-in points — we draw them ourselves below)
    fig6.add_trace(go.Box(
        y=subset["hog_size_expressed"],
        x=[x_pos] * len(subset),
        name=cat,
        marker_color=color,
        line_color=color,
        fillcolor=color,
        opacity=0.6,
        boxpoints=False,
        width=0.5,
        showlegend=False
    ))

    # Regular points (size > 2) — circles
    reg = subset[subset["hog_size_expressed"] != 2]
    if not reg.empty:
        fig6.add_trace(go.Scatter(
            x=x_pos + np.random.uniform(-0.15, 0.15, len(reg)),
            y=reg["hog_size_expressed"],
            mode="markers",
            marker=dict(color=color, symbol="circle", size=5, opacity=0.7,
                        line=dict(width=0.5, color="white")),
            showlegend=False,
            text=reg["HOG"].astype(str),
            hovertemplate="HOG: %{text}<br>Size: %{y}<extra></extra>"
        ))

    # HOG size 2 shape — diamond (All three cannot exist at size 2)
    s2 = subset[subset["hog_size_expressed"] == 2]
    if not s2.empty:
        fig6.add_trace(go.Scatter(
            x=x_pos + np.random.uniform(-0.15, 0.15, len(s2)),
            y=s2["hog_size_expressed"],
            mode="markers",
            marker=dict(color=color, symbol="diamond", size=8, opacity=0.9,
                        line=dict(width=0.8, color="white")),
            showlegend=False,
            text=s2["HOG"].astype(str),
            hovertemplate="HOG: %{text}<br>Size: 2 ◆<extra></extra>"
        ))

fig6.update_layout(
    title=dict(text="(expressed) Gene family sizes in different transcript bias composition categories", x=0.5, xanchor="center"),
    xaxis=dict(
        tickmode="array",
        tickvals=list(cat_positions.values()),
        ticktext=list(cat_positions.keys()),
        tickangle=45,
        title="Gene family composition category"
    ),
    yaxis_title="Gene family size (expressed)",
    showlegend=False
)

fig6.show()

In [75]:

hog_dir_melted = hog_direction.melt(
    id_vars=["HOG", "category"],
    value_vars=["hog_size_genome", "hog_size_mapped", "hog_size_expressed"],
    var_name="size_level",
    value_name="HOG_size"
)

size_level_labels = {
    "hog_size_genome":    "Genome",
    "hog_size_mapped":    "Mapped",
    "hog_size_expressed": "Expressed",
}
hog_dir_melted["size_level"] = hog_dir_melted["size_level"].map(size_level_labels)

fig7 = px.box(
    hog_dir_melted,
    x="category",
    y="HOG_size",
    color="size_level",
    color_discrete_map={
        "Genome":    "#9E9E9E",
        "Mapped":    "#C4A35A",
        "Expressed": "#5BA67A",
    },
    category_orders={
        "category":   category_order,
        "size_level": ["Genome", "Mapped", "Expressed"],
    },
    points="all",
    hover_data=["HOG"],
)

fig7.update_layout(
    title=dict(text="Gene family sizes in different transcript bias composition categories, across size levels", x=0.5, xanchor="center"),
    xaxis_title="Gene family composition category",
    yaxis_title="Gene family size",
    xaxis_tickangle=45,
    legend_title_text="Size level",
)
fig7.show()


In [76]:
fig8 = px.histogram(
    hog_direction,
    x="hog_size_genome",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    barmode="group"
)
fig8.update_xaxes(
    tickmode="linear",
    dtick=10
)
fig8.update_yaxes(
    type="log",
    tickmode="array",
    tickvals=[1, 10, 100, 1000],
    ticktext=["1", "10", "100", "1000"]
)

fig8.update_layout(
    title=dict(text="Number of gene families in each size and transcript bias composition categories", x=0.5, xanchor="center"),
    xaxis_title="Gene family size (genome wide)",
    yaxis_title="Number of Gene families (log10 scale)"
)

fig8.show()

y axis log scaled. line plot for each category instead. 

Same but line plot

In [77]:
import pandas as pd

# Get full range of sizes
all_sizes = range(hog_direction["hog_size_genome"].min(),
                  hog_direction["hog_size_genome"].max() + 1)

# Create full grid
full_index = pd.MultiIndex.from_product(
    [all_sizes, category_order],
    names=["HOG_size", "category"]
)

hog_size_counts = (
    hog_direction
    .groupby(["hog_size_genome", "category"], observed=False)
    .size()
    .reindex(full_index, fill_value=0)
    .reset_index(name="count")
)

#with log scale we cannot see 0. 
hog_size_counts["count_plot"] = hog_size_counts["count"].replace(0, 0.1)

fig9 = px.line(
    hog_size_counts,
    x="HOG_size",
    y="count_plot",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    markers=True,
)

fig9.update_xaxes(
    tickmode="linear",
    dtick=10
)

fig9.update_yaxes(
    type="log",
    tickmode="array",
    tickvals=[1, 10, 100, 1000],
    ticktext=["1", "10", "100", "1000"]
)

fig9.update_layout(
    title=dict(text="Number of gene families in each size and transcript bias composition categories", x=0.5, xanchor="center"),
    xaxis_title="Gene family size",
    yaxis_title="Number of Gene families (log scale)"
)

fig9.show()

% proportion bar chart

In [78]:
hog_size_cat = (
    hog_direction
    .groupby(["hog_size_expressed", "category"], observed=False)
    .size()
    .reset_index(name="count")
)

size_totals = (
    hog_size_cat.groupby("hog_size_expressed")["count"]
    .sum()
    .reset_index(name="total")
)

hog_size_cat = hog_size_cat.merge(size_totals, on="hog_size_expressed")
hog_size_cat["pct"] = hog_size_cat["count"] / hog_size_cat["total"] * 100
hog_size_cat["label"] = hog_size_cat["count"].where(hog_size_cat["pct"] >= 4, other="")

fig10 = px.bar(
    hog_size_cat,
    x="hog_size_expressed",
    y="pct",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    text="label",
    custom_data=["count", "total", "category"],
    barmode="stack"
)

fig10.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=10),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "HOG size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)

# Total n= annotations below each bar
annotations = [
    dict(
        x=row.hog_size_expressed, y=101,
        xref="x", yref="y",
        text=f"n={row.total}",
        showarrow=False,
        font=dict(size=11, color="#444"),
        yanchor="bottom",
        textangle=-45
    )
    for row in size_totals.itertuples()
]

fig10.update_layout(
    title=dict(text="Proportion of gene family categories in each size", x=0.5, xanchor="center"),
    xaxis=dict(tickmode="linear", dtick=1, title="Gene family size (expressed)"),
    yaxis=dict(title="Proportion of gene families (%)", range=[0, 111], ticksuffix="%"),
    legend_title_text="Category",
    annotations=annotations,
    margin=dict(b=60),
    width=1100,
    height=600
)

fig10.show()

In [79]:
hog_size_cat = (
    hog_direction
    .groupby(["hog_size_genome", "category"], observed=False)
    .size()
    .reset_index(name="count")
)

size_totals = (
    hog_size_cat.groupby("hog_size_genome")["count"]
    .sum()
    .reset_index(name="total")
)

hog_size_cat = hog_size_cat.merge(size_totals, on="hog_size_genome")
hog_size_cat["pct"] = hog_size_cat["count"] / hog_size_cat["total"] * 100
hog_size_cat["label"] = hog_size_cat["count"].where(hog_size_cat["pct"] >= 4, other="")

fig10 = px.bar(
    hog_size_cat,
    x="hog_size_genome",
    y="pct",
    color="category",
    color_discrete_map=category_colors,
    category_orders={"category": category_order},
    text="label",
    custom_data=["count", "total", "category"],
    barmode="stack"
)

fig10.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=10),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "Gene family size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)

# Total n= annotations below each bar
annotations = [
    dict(
        x=row.hog_size_genome, y=101,
        xref="x", yref="y",
        text=f"n={row.total}",
        showarrow=False,
        font=dict(size=9, color="#444"),
        yanchor="bottom",
        textangle=-80
    )
    for row in size_totals.itertuples()
]

fig10.update_layout(
    title=dict(text="Proportion of gene family categories in each size (genome)", x=0.5, xanchor="center"),
    xaxis=dict(tickmode="linear", dtick=5, title="Gene family size (genome, sizes ≥ 2)"),
    yaxis=dict(title="Proportion of gene families (%)", range=[0, 111], ticksuffix="%"),
    legend=dict(
    title="Category",
    orientation="h",
    yanchor="bottom", y=1.02,
    xanchor="right", x=1,
),
    annotations=annotations,
    margin=dict(b=60, t=120),
    width=1200,
    height=600
)
fig10.write_image("proportions_categories_size.svg")
fig10.show()

Density curve plot

In [80]:
from scipy.stats import gaussian_kde

sizes = hog_direction["hog_size_expressed"]
x_range = np.linspace(sizes.min(), sizes.max(), 300)

kde = gaussian_kde(sizes)
y_vals = kde(x_range) * len(sizes)

fig11 = go.Figure(go.Scatter(
    x=x_range,
    y=y_vals,
    mode="lines",
    fill="tozeroy",
    line=dict(color="#0030FF", width=2),
    fillcolor="rgba(0, 48, 255, 0.2)"
))

fig11.update_layout(
    title=dict(text="density curve of expressed transcripts", x=0.5, xanchor="center"),
    xaxis=dict(title="Gene family size", tickmode="linear", dtick=1),
    yaxis_title="Number of gene families",
    width=1000,
    height=500
)

fig11.show()

Data from prefiltered/unexpressed transcripts. These are post tximport in R, so they are transcrtipts that were still mapped by Salmon. The raw orthofinder data includes even more transcripts, but these have no mapping evidence.

In [81]:
def get_independent_sizes(df, hog_col="HOG"):
    return (
        df.dropna(subset=[hog_col])
        .groupby(hog_col)
        .size()
    )

sizes_genome    = get_independent_sizes(full_annotation)
sizes_mapped    = get_independent_sizes(prefilter_df)
sizes_expressed = get_independent_sizes(salmon_map_results_HOG)

density_datasets = {
    f"Genome (n={( sizes_genome    >= 2).sum():,} Gene families)": (sizes_genome[sizes_genome >= 2],       "#9E9E9E", "rgba(158,158,158,0.2)"),
    f"Mapped (n={(sizes_mapped     >= 2).sum():,} Gene families)": (sizes_mapped[sizes_mapped >= 2],       "#C4A35A", "rgba(196,163,90,0.2)"),
    f"Expressed (n={(sizes_expressed >= 2).sum():,} Gene families)": (sizes_expressed[sizes_expressed >= 2], "#5BA67A", "rgba(91,166,122,0.2)"),
}

fig12 = go.Figure()

for label, (sizes, color, fillcolor) in density_datasets.items():
    counts = sizes.value_counts().sort_index()
    fig12.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
    ))

fig12.update_layout(
    title=dict(
        text="Gene family size distribution across dataset levels (size ≥ 2)",
        x=0.5, xanchor="center"
    ),
    xaxis=dict(title="Gene family size", tickmode="linear", dtick=5),
    yaxis=dict(title="Number of gene families"),
    legend_title_text="Dataset level",
    plot_bgcolor="white",
    width=800,
    height=400,
)
fig12.show()

Log scaled

In [82]:
def get_independent_sizes(df, hog_col="HOG"):
    return (
        df.dropna(subset=[hog_col])
        .groupby(hog_col)
        .size()
    )

sizes_genome    = get_independent_sizes(full_annotation)
sizes_mapped    = get_independent_sizes(prefilter_df)
sizes_expressed = get_independent_sizes(salmon_map_results_HOG)

density_datasets = {
    f"Genome (n={( sizes_genome    >= 2).sum():,} Gene families)": (sizes_genome[sizes_genome >= 2],       "#9E9E9E", "rgba(158,158,158,0.2)"),
    f"Mapped (n={(sizes_mapped     >= 2).sum():,} Gene families)": (sizes_mapped[sizes_mapped >= 2],       "#C4A35A", "rgba(196,163,90,0.2)"),
    f"Expressed (n={(sizes_expressed >= 2).sum():,} Gene families)": (sizes_expressed[sizes_expressed >= 2], "#5BA67A", "rgba(91,166,122,0.2)"),
}

fig13 = go.Figure()

for label, (sizes, color, fillcolor) in density_datasets.items():
    counts = sizes.value_counts().sort_index()
    fig13.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
    ))

fig13.update_layout(
    title=dict(
        text="Gene family size distribution across dataset levels (size ≥ 2)",
        x=0.5, xanchor="center"
    ),
    xaxis=dict(title="Gene family size", tickmode="linear", dtick=5),
    yaxis=dict(
        title="Number of gene families (log-scaled)",
        type="log",
        tickmode="array",
        tickvals=[1, 10, 100, 1000, 10000],
        ticktext=["1", "10", "100", "1000", "10000"]
        ),
    legend_title_text="Dataset level",
    plot_bgcolor="white",
    width=800,
    height=400,
)

fig13.show()


Subplots for convenience

In [83]:
from plotly.subplots import make_subplots

def get_independent_sizes(df, hog_col="HOG"):
    return (
        df.dropna(subset=[hog_col])
        .groupby(hog_col)
        .size()
    )

sizes_genome    = get_independent_sizes(full_annotation)
sizes_mapped    = get_independent_sizes(prefilter_df)
sizes_expressed = get_independent_sizes(salmon_map_results_HOG)

density_datasets = {
    f"Genome (n={( sizes_genome    >= 2).sum():,} Gene families)": (sizes_genome[sizes_genome >= 2],       "#9E9E9E", "rgba(158,158,158,0.2)"),
    f"Mapped (n={(sizes_mapped     >= 2).sum():,} Gene families)": (sizes_mapped[sizes_mapped >= 2],       "#C4A35A", "rgba(196,163,90,0.2)"),
    f"Expressed (n={(sizes_expressed >= 2).sum():,} Gene families)": (sizes_expressed[sizes_expressed >= 2], "#5BA67A", "rgba(91,166,122,0.2)"),
}

fig = make_subplots(
    rows=1, cols=2,
    shared_xaxes=False,
    subplot_titles=("Linear scale", "Log scale"),
    horizontal_spacing=0.12,
)

for i, (label, (sizes, color, fillcolor)) in enumerate(density_datasets.items()):
    counts = sizes.value_counts().sort_index()

    # Linear subplot
    fig.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
        legendgroup=label,
        showlegend=True,
    ), row=1, col=1)

    # Log subplot — reuse same legend group, hide duplicate legend entry
    fig.add_trace(go.Scatter(
        x=counts.index,
        y=counts.values,
        mode="lines+markers",
        name=label,
        fill="tozeroy",
        line=dict(color=color, width=2),
        fillcolor=fillcolor,
        marker=dict(size=4),
        legendgroup=label,
        showlegend=False,
    ), row=1, col=2)

fig.update_xaxes(title_text="Gene family size", tickmode="linear", dtick=5)

fig.update_yaxes(title_text="Number of gene families", row=1, col=1)
fig.update_yaxes(
    title_text="Number of gene families (log-scaled)",
    type="log",
    tickmode="array",
    tickvals=[1, 10, 100, 1000, 10000],
    ticktext=["1", "10", "100", "1,000", "10,000"],
    row=1, col=2,
)

fig.update_layout(
    title=dict(
        text="Gene family size distribution across dataset levels (size ≥ 2)",
        x=0.5, xanchor="center"
    ),
    legend_title_text="Dataset level",
    plot_bgcolor="white",
    width=1400,
    height=450,
)
fig.write_image("size_distribution_subplot.svg")
fig.show()

# Expressed vs unexpressed mapped transcripts per HOG size 

In [84]:
# build the transcript level classification from full_annotation
mapped_ids = set(prefilter_df["transcript_id"])
expressed_ids = set(salmon_map_results_HOG["transcript_id"])

annot = full_annotation.dropna(subset=["HOG"]).copy()
annot["hog_size_genome"] = annot.groupby("HOG")["HOG"].transform("count")

def classify_level(tid):
    if tid in expressed_ids:
        return "Expressed"
    elif tid in mapped_ids:
        return "Mapped, not expressed"
    else: 
        return "Not mapped"

annot["level"] = annot["transcript_id"].apply(classify_level)

level_order = ["Expressed", "Mapped, not expressed", "Not mapped"]
level_colors = {
    "Expressed":              "#5BA67A",
    "Mapped, not expressed":  "#C4A35A",
    "Not mapped":             "#9E9E9E",
}

# Compute proportions per genome gene family size
level_counts = (
    annot[annot["hog_size_genome"] >= 2]
    .groupby(["hog_size_genome", "level"])
    .size()
    .reset_index(name="count")
)

size_totals = level_counts.groupby("hog_size_genome")["count"].sum().reset_index(name="total")
level_counts = level_counts.merge(size_totals, on="hog_size_genome")
level_counts["pct"]   = level_counts["count"] / level_counts["total"] * 100
level_counts["label"] = level_counts["count"].where(level_counts["pct"] >= 4, other="")

fig15 = px.bar(
    level_counts,
    x="hog_size_genome",
    y="pct",
    color="level",
    color_discrete_map=level_colors,
    category_orders={"level": level_order},
    text="label",
    custom_data=["count", "total", "level"],
    barmode="stack"
)

fig15.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=11),
    hovertemplate=(
        "<b>%{customdata[2]}</b><br>"
        "Gene family size: %{x}<br>"
        "Proportion: %{y:.1f}%<br>"
        "Count: %{customdata[0]}<br>"
        "Total at this size: %{customdata[1]}"
        "<extra></extra>"
    )
)

annotations = [
    dict(
        x=row.hog_size_genome, y=101,
        xref="x", yref="y",
        text=f"<i>n={row.total}</i>",
        showarrow=False,
        font=dict(size=8, color="#444"),
        yanchor="bottom",
        textangle=-80
    )
    for row in size_totals.itertuples()
]

fig15.update_layout(
    title=dict(
        text="Transcript expression level by gene family size (genome)",
        x=0.5, xanchor="center"
    ),
    xaxis=dict(tickmode="linear", dtick=5, title="Gene family size (genome, sizes ≥ 2)"),
    yaxis=dict(title="Proportion of transcripts (%)", range=[0, 110], ticksuffix="%"),
    legend=dict(
        title="Level: ",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
    annotations=annotations,
    margin=dict(b=80, t=120),
    width=1300,
    height=600,
    plot_bgcolor="white",
)

# add mean line for the expressed transcripts. This is based on the weighted mean
expressed_rows = level_counts[level_counts["level"] == "Expressed"]
#mean_expressed_pct_weighted = expressed_rows["count"].sum() / expressed_rows["total"].sum() * 100
mean_expressed_pct = expressed_rows["pct"].mean()

fig15.add_hline(
    y=mean_expressed_pct,
    line_color="red",
    line_width=1.5,
    line_dash="dash",
    annotation_text=f"Mean expressed: {mean_expressed_pct:.1f}%",
    annotation_position="top right",
    annotation_font=dict(color="red", size=11),
)
fig15.write_image("proportion_expression_sizes.svg")
fig15.show()
#these size numbers comes from the total unexpressed dataset and are not comparable to the filtered hog_size_table above. 

# Plot each transcript vs. HOG size 

In [85]:
salmon_map_results_HOG_filtered = salmon_map_results_HOG[salmon_map_results_HOG["hog_size_genome"] >= 2].copy()

np.random.seed(42)
jitter = np.random.uniform(-0.15, 0.15, size=len(salmon_map_results_HOG_filtered))

fig17 = px.scatter(
    salmon_map_results_HOG_filtered,
    x=salmon_map_results_HOG_filtered["hog_size_genome"] + jitter,
    y="log2FoldChange",
    opacity=0.4,
    color="sex_bias",
    color_discrete_map={
        "male":            "#6BAED6",
        "female":          "#E07B8A",
        "not_significant": "#9E9E9E",
    },
    hover_data={"transcript_id": True, "HOG": True, "hog_size_expressed": True},
    labels={
        "x":             "Gene family size (genome)",
        "log2FoldChange": "Transcript log2FoldChange",
        "hog_size_expressed": "Gene family size (expressed)"
    },
    title="Transcript-level sex bias as a function of gene family size (genome)"
)

fig17.add_hline(y=0, line_dash="dash", line_color="black")
fig17.update_layout(
    xaxis=dict(tickmode="linear", dtick=5, title="Gene family size (genome, size ≥ 2)"),
    plot_bgcolor="white",
    legend_title_text="Sex bias",
    width=1200
    
)
fig17.write_image("transcript_expression_size.svg")
fig17.show()